In [5]:
# DBTokenizer Demo

# This notebook demonstrates the DBTokenizer on synthetic clinical data for two patients.

# Data classes:
# - demographic: birth date
# - lab: hemoglobin, glucose, creatinine
# - vital: heart rate, blood pressure, SpO2
# - medication: compound, route, frequency, units as text; dose as numeric
# - flowsheet: categorical observations (O2 device, diet, fall risk)
# - admission / discharge: hospital stay boundaries

# We run two configurations:
# 1. BPE + Discrete + Factored: numeric values are binned into discrete tokens, placed as separate tokens after the text token
# 2. BPE + Continuous + Fused: numeric values are z-scored and paired in a parallel float array; concept-mode text+level tokens are fused into single compound tokens

In [6]:
import polars as pl
import numpy as np
from tokenizer import DBTokenizer

In [ ]:
## Load Demo Data

In [7]:
df = pl.read_csv(
    "data/demo.csv",
    try_parse_dates=True,
    null_values=[""],
)
print(f"Loaded {df.height} rows for {df['id'].n_unique()} patients")
print(f"Classes: {sorted(df['class'].unique().to_list())}")
df.head(20)

Loaded 106 rows for 2 patients
Classes: ['admission', 'demographic', 'discharge', 'flowsheet', 'lab', 'medication', 'vital']


id,time,class,text_value,numeric_value
i64,datetime[μs],str,str,f64
1,1958-03-15 00:00:00,"""demographic""","""birth_date""",null
1,2024-01-02 08:00:00,"""lab""","""hemoglobin""",13.2
1,2024-01-02 08:00:00,"""lab""","""glucose""",98.0
1,2024-01-02 08:00:00,"""lab""","""creatinine""",0.9
1,2024-01-02 08:05:00,"""vital""","""heart_rate""",78.0
…,…,…,…,…
1,2024-01-15 09:30:00,"""lab""","""creatinine""",1.0
1,2024-01-15 09:35:00,"""vital""","""heart_rate""",82.0
1,2024-01-15 09:35:00,"""vital""","""blood_pressure_systolic""",132.0


In [8]:
# Configuration 1: BPE + Discrete + Factored

# This is the simplest BPE configuration:
# - text_mode_default="auto": classes with many unique text values (like medication) get BPE; low-cardinality classes (lab, vital, flowsheet) get concept tokens
# - num_type="discrete": numeric values are quantile-binned into `<|Q0|>` … `<|Q{n}|>` tokens (or `<|L0|>` … for ≤10 distinct values)
# - num_seq="factored": numeric tokens are emitted as separate tokens after the text token

In [21]:
tok1 = DBTokenizer(
    text_mode_default="auto",
    text_mode_threshold=4,
    num_type="discrete",
    num_seq="factored",
    n_bins=5,
    final_vocab_size=512,
    admission_classes=["admission"],
    discharge_classes=["discharge"],
    milestone_ip="8hr",
    milestone_op="week",
)
tok1.train(df)

Extracting birth dates from data...
  Found birth dates for 2 patients
Resolving per-class text modes...
  4 concept classes, 2 BPE classes, 0 pair-level overrides
Training numeric transforms...
  13 numeric groups: 11 level, 2 bins, 0 scaling
Training time-delta transforms...
  IP time-delta: bins (-)
  OP time-delta: bins (-)
Building vocabulary...
  BPE vocab size: 512 (237 special/concept, 275 BPE merges)
Training complete.


In [22]:
print("Resolved text modes per class:")
for cls, mode in sorted(tok1.class_text_modes.items()):
    print(f"  {cls:20s} -> {mode}")

Resolved text modes per class:
  admission            -> concept
  discharge            -> concept
  flowsheet            -> bpe
  lab                  -> concept
  medication           -> bpe
  vital                -> concept


In [23]:
ids1, vals1 = tok1.encode(df)
print(f"Token count: {len(ids1)}")
print(f"vals is None (discrete mode): {vals1 is None}")
print()

# Decode back to human-readable tokens
decoded1 = tok1.decode(ids1)
print("First 60 tokens:")
for i, t in enumerate(decoded1[:60]):
    print(f"  [{i:3d}] {t}")

Token count: 820
vals is None (discrete mode): True

First 60 tokens:
  [  0] <|sos|>
  [  1] <|age_0|>
  [  2] <|age_65|>
  [  3] <|lab|>
  [  4] hemoglobin
  [  5] <|L7|>
  [  6] <|lab|>
  [  7] glucose
  [  8] <|L2|>
  [  9] <|lab|>
  [ 10] creatinine
  [ 11] <|L1|>
  [ 12] <|delta_time_op|>
  [ 13] <|Q2|>
  [ 14] <|ms_week_1|>
  [ 15] <|vital|>
  [ 16] heart_rate
  [ 17] <|Q2|>
  [ 18] <|vital|>
  [ 19] blood_pressure_systolic
  [ 20] <|Q2|>
  [ 21] <|vital|>
  [ 22] blood_pressure_diastolic
  [ 23] <|L3|>
  [ 24] <|vital|>
  [ 25] spo2
  [ 26] <|L5|>
  [ 27] <|delta_time_op|>
  [ 28] <|Q2|>
  [ 29] <|medication|>
  [ 30] l
  [ 31] is
  [ 32] in
  [ 33] o
  [ 34] p
  [ 35] r
  [ 36] i
  [ 37] l
  [ 38] ,
  [ 39]  oral
  [ 40] ,
  [ 41]  
  [ 42] d
  [ 43] ai
  [ 44] l
  [ 45] y
  [ 46] ,
  [ 47]  mg
  [ 48] <|L0|>
  [ 49] <|medication|>
  [ 50] m
  [ 51] e
  [ 52] t
  [ 53] f
  [ 54] or
  [ 55] m
  [ 56] in
  [ 57] ,
  [ 58]  oral
  [ 59] ,


In [24]:
# Debug view: see per-row token breakdown
dbg1 = tok1.encode_debug(df)
print("\nDebug view of first 10 rows:")
dbg1.head(10)


Debug view of first 10 rows:


id,time,class,text_value,numeric_value,tokens
i64,datetime[μs],str,str,f64,list[str]
1,1958-03-15 00:00:00,"""demographic""","""birth_date""",null,"[""<|sos|>"", ""<|age_0|>""]"
1,2024-01-02 08:00:00,"""lab""","""hemoglobin""",13.2,"[""<|age_65|>"", ""<|lab|>"", … ""<|L7|>""]"
1,2024-01-02 08:00:00,"""lab""","""glucose""",98.0,"[""<|lab|>"", ""glucose"", ""<|L2|>""]"
1,2024-01-02 08:00:00,"""lab""","""creatinine""",0.9,"[""<|lab|>"", ""creatinine"", ""<|L1|>""]"
1,2024-01-02 08:05:00,"""vital""","""heart_rate""",78.0,"[""<|delta_time_op|>"", ""<|Q2|>"", … ""<|Q2|>""]"
1,2024-01-02 08:05:00,"""vital""","""blood_pressure_systolic""",128.0,"[""<|vital|>"", ""blood_pressure_systolic"", ""<|Q2|>""]"
1,2024-01-02 08:05:00,"""vital""","""blood_pressure_diastolic""",82.0,"[""<|vital|>"", ""blood_pressure_diastolic"", ""<|L3|>""]"
1,2024-01-02 08:05:00,"""vital""","""spo2""",97.0,"[""<|vital|>"", ""spo2"", ""<|L5|>""]"
1,2024-01-02 08:10:00,"""medication""","""lisinopril, oral, daily, mg""",10.0,"[""<|delta_time_op|>"", ""<|Q2|>"", … ""<|L0|>""]"


In [ ]:
# # Configuration 2: BPE + Continuous + Fused

# This configuration demonstrates:
# - num_type="continuous": numeric values are distribution-scaled to z-scores, stored in a parallel `vals` array
# - num_seq="fused": for concept-mode text groups, the text token and its level are combined into a single compound token (e.g. `hemoglobin::L4`)

# ### Important: Concept Overrides for Fused Numerics

# Fused mode only works with concept text tokens (not BPE). For classes that have true numeric measurements you want fused with levels, you must ensure they use concept mode.

# Since our lab and vital classes have low cardinality, `auto` already assigns them concept mode. But if any class were incorrectly assigned BPE, you'd override it:
# text_mode_overrides={"lab": "concept", "vital": "concept"}

# BPE classes automatically fall back to factored numeric placement (with a warning).

In [25]:
tok2 = DBTokenizer(
    text_mode_default="auto",
    text_mode_threshold=16,
    # Explicitly ensure numeric-bearing classes use concept mode for fusion
    text_mode_overrides={
        "lab": "concept",
        "vital": "concept",
        "flowsheet": "concept",
    },
    num_type="continuous",
    num_seq="fused",
    n_bins=5,
    final_vocab_size=512,
    admission_classes=["admission"],
    discharge_classes=["discharge"],
    milestone_ip="8hr",
    milestone_op="week",
)
tok2.train(df)

Extracting birth dates from data...
  Found birth dates for 2 patients
Resolving per-class text modes...
  6 concept classes, 0 BPE classes, 0 pair-level overrides
Training numeric transforms...
  13 numeric groups: 11 level, 0 bins, 2 scaling
Training time-delta transforms...
  IP time-delta: scaling (gamma)
  OP time-delta: scaling (gamma)
Building vocabulary...
  Concept vocab size: 302
Training complete.


/Users/andrewloza/Documents/Research/dbtoken/tokenizer.py:440: UserWarning: Distribution auto-detect failed: unable to find column "count"; valid columns: []. Falling back to normal.
  warnings.warn(


In [26]:
print("Resolved text modes per class:")
for cls, mode in sorted(tok2.class_text_modes.items()):
    print(f"  {cls:20s} -> {mode}")

print()
print("Numeric parameter types:")
for key, params in sorted(tok2.numeric_params.items(), key=str):
    cls, tv = key
    print(f"  ({cls}, {tv}): {params['type']}")

Resolved text modes per class:
  admission            -> concept
  discharge            -> concept
  flowsheet            -> concept
  lab                  -> concept
  medication           -> concept
  vital                -> concept

Numeric parameter types:
  (lab, creatinine): level
  (lab, glucose): level
  (lab, hemoglobin): level
  (medication, atorvastatin, oral, daily, mg): level
  (medication, furosemide, IV, BID, mg): level
  (medication, heparin, IV, continuous, units): level
  (medication, lisinopril, oral, daily, mg): level
  (medication, metformin, oral, BID, mg): level
  (medication, vancomycin, IV, q12h, mg): level
  (vital, blood_pressure_diastolic): level
  (vital, blood_pressure_systolic): scaling
  (vital, heart_rate): scaling
  (vital, spo2): level


In [27]:
ids2, vals2 = tok2.encode(df)
print(f"Token count: {len(ids2)}")
print(f"vals length:  {len(vals2)}")
print()

# Show tokens alongside their continuous values
decoded2 = tok2.decode(ids2)
print("First 60 token-value pairs:")
for i, (t, v) in enumerate(zip(decoded2[:60], vals2[:60])):
    v_str = f"{v:8.3f}" if not np.isnan(v) else "     NaN"
    print(f"  [{i:3d}] {v_str}  {t}")

Token count: 265
vals length:  265

First 60 token-value pairs:
  [  0]      NaN  <|sos|>
  [  1]      NaN  <|age_0|>
  [  2]      NaN  <|age_65|>
  [  3]      NaN  <|lab|>
  [  4]      NaN  hemoglobin::L7
  [  5]      NaN  <|lab|>
  [  6]      NaN  glucose::L2
  [  7]      NaN  <|lab|>
  [  8]      NaN  creatinine::L1
  [  9]   -0.527  <|delta_time_op|>
  [ 10]      NaN  <|ms_week_1|>
  [ 11]      NaN  <|vital|>
  [ 12]   -0.557  heart_rate
  [ 13]      NaN  <|vital|>
  [ 14]   -0.555  blood_pressure_systolic
  [ 15]      NaN  <|vital|>
  [ 16]      NaN  blood_pressure_diastolic::L3
  [ 17]      NaN  <|vital|>
  [ 18]      NaN  spo2::L5
  [ 19]   -0.527  <|delta_time_op|>
  [ 20]      NaN  <|medication|>
  [ 21]      NaN  lisinopril, oral, daily, mg::L0
  [ 22]      NaN  <|medication|>
  [ 23]      NaN  metformin, oral, BID, mg::L0
  [ 24]   -0.527  <|delta_time_op|>
  [ 25]      NaN  <|flowsheet|>
  [ 26]      NaN  o2_device: room_air
  [ 27]      NaN  <|flowsheet|>
  [ 28]      NaN 

In [28]:
# Debug view: fused tokens
dbg2 = tok2.encode_debug(df)
print("Per-row token breakdown (first 20 rows):")
for row in dbg2.head(20).iter_rows(named=True):
    print(f"  id={row['id']} {row['class']:15s} {str(row['text_value']):35s} "
          f"num={row['numeric_value']}")
    print(f"    -> {row['tokens']}")

Per-row token breakdown (first 20 rows):
  id=1 demographic     birth_date                          num=None
    -> ['<|sos|>', '<|age_0|>']
  id=1 lab             hemoglobin                          num=13.2
    -> ['<|age_65|>', '<|lab|>', 'hemoglobin::L7']
  id=1 lab             glucose                             num=98.0
    -> ['<|lab|>', 'glucose::L2']
  id=1 lab             creatinine                          num=0.9
    -> ['<|lab|>', 'creatinine::L1']
  id=1 vital           heart_rate                          num=78.0
    -> ['<|delta_time_op|>(-0.527)', '<|ms_week_1|>', '<|vital|>', 'heart_rate(-0.557)']
  id=1 vital           blood_pressure_systolic             num=128.0
    -> ['<|vital|>', 'blood_pressure_systolic(-0.555)']
  id=1 vital           blood_pressure_diastolic            num=82.0
    -> ['<|vital|>', 'blood_pressure_diastolic::L3']
  id=1 vital           spo2                                num=97.0
    -> ['<|vital|>', 'spo2::L5']
  id=1 medication      lisinop

## Comparison

Let's compare the two configurations side by side.

In [29]:
print(f"Configuration 1 (BPE + Discrete + Factored):")
print(f"  Total tokens: {len(ids1)}")
print(f"  Vocab size:   {tok1.final_vocab_size}")
print(f"  vals array:   {'None' if vals1 is None else len(vals1)}")
print()
print(f"Configuration 2 (BPE + Continuous + Fused):")
print(f"  Total tokens: {len(ids2)}")
print(f"  Vocab size:   {tok2.final_vocab_size}")
print(f"  vals array:   {len(vals2)}")
print()
print(f"Fused mode reduces sequence length by "
      f"{len(ids1) - len(ids2)} tokens ({(1 - len(ids2)/len(ids1))*100:.1f}%)")

Configuration 1 (BPE + Discrete + Factored):
  Total tokens: 820
  Vocab size:   512
  vals array:   None

Configuration 2 (BPE + Continuous + Fused):
  Total tokens: 265
  Vocab size:   302
  vals array:   265

Fused mode reduces sequence length by 555 tokens (67.7%)


## Save & Load Round-Trip

In [30]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "demo_tokenizer")
    tok2.save(path)
    tok2_loaded = DBTokenizer.load(path)

    ids2_rt, vals2_rt = tok2_loaded.encode(df)
    assert ids2 == ids2_rt, "Token ID mismatch after round-trip!"
    assert len(vals2) == len(vals2_rt), "Vals length mismatch!"
    print("Save/load round-trip: PASSED ✓")

Saved tokenizer to /var/folders/jx/qpq3p10j7nz660bbfmr0pkrm0000gn/T/tmpnqumec76/demo_tokenizer.json
Loaded tokenizer from /var/folders/jx/qpq3p10j7nz660bbfmr0pkrm0000gn/T/tmpnqumec76/demo_tokenizer.json
Save/load round-trip: PASSED ✓
